# Phase 9-C Step 3: 3システム比較評価

**作成日**: 2026-03-01  
**プロジェクト**: experiments-local-llm  
**目的**: FT単体・RAG単体・FT+RAG の効果を分離測定する

---

## 比較する 3 システム

| System | 構成 | 目的 |
|--------|------|------|
| **A: RAG (C2)** | Qwen3-32B + RAG | ベースライン（既存結果を使用） |
| **B: FT-only** | QLoRA Qwen3-32B + システムプロンプト（RAGなし） | FT単体の効果測定 |
| **C: FT+RAG** | QLoRA Qwen3-32B + RAG | FT と RAG の相補効果測定 |

## C3 目標値

| 指標 | C2 実績 | C3 目標 |
|------|---------|--------|
| composite_score | 70.4 | **75+** |
| reasoning_score | 3.07 | **3.5+** |
| evidence_score | 3.85 | **4.0+** |
| composite_success_rate | 83.1% | **88%+** |

## 0. リポジトリ同期

In [ ]:
import os
import sys

if 'google.colab' in sys.modules:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')

REPO_PATH = '/content/drive/MyDrive/experiments-local-llm'
BRANCH = 'feature/phase9c-step3-qlora'

print(f"Repository: {REPO_PATH}")
print(f"Target branch: {BRANCH}")
print()

from getpass import getpass
github_token = getpass("GitHub Personal Access Token: ")
!git -C {REPO_PATH} remote set-url origin https://{github_token}@github.com/mopinfish/experiments-local-llm.git
print("GitHub HTTPS authentication configured\n")

!git -C {REPO_PATH} config user.name "colab-runner"
!git -C {REPO_PATH} config user.email "colab@example.com"
!git -C {REPO_PATH} stash --include-untracked -m "auto-stash before sync"
!git -C {REPO_PATH} fetch origin
!git -C {REPO_PATH} checkout {BRANCH}
!git -C {REPO_PATH} pull origin {BRANCH}
!git -C {REPO_PATH} stash pop 2>/dev/null || echo "No stash to pop"

print()
print("=" * 60)
!git -C {REPO_PATH} log --oneline -5
!git -C {REPO_PATH} branch --show-current
print("=" * 60)
print("\n\u2705 リポジトリ同期完了")

## 1. 環境セットアップ

In [ ]:
import sys
import os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    PROJECT_PATH = '/content/drive/MyDrive/experiments-local-llm'
    sys.path.insert(0, f'{PROJECT_PATH}/src')

    !pip install -q chromadb sentence-transformers networkx
    !pip install -q transformers accelerate bitsandbytes
    !pip install -q peft
    !pip install -q langchain langchain-core langchain-community langchain-chroma langgraph
    !pip install -q matplotlib seaborn pandas numpy tqdm
else:
    PROJECT_PATH = '..'
    sys.path.insert(0, f'{PROJECT_PATH}/src')

os.makedirs(f'{PROJECT_PATH}/results', exist_ok=True)

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram_gb:.1f} GB")

    if 'A100' not in gpu_name:
        print(f"\n\u26a0\ufe0f WARNING: A100 GPU required!")
        print(f"  Current GPU: {gpu_name}")
        raise RuntimeError(f"A100 GPU required, got {gpu_name}")
    else:
        print("\u2705 A100 GPU confirmed")

print(f"\nProject path: {PROJECT_PATH}")

## 2. FTモデルロード (Qwen3-32B + LoRA)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import gc
import re
import json
import warnings
warnings.filterwarnings('ignore')

# VRAMクリーンアップ
for var_name in ['model', 'ft_model', 'tokenizer']:
    if var_name in dir():
        try:
            del globals()[var_name]
        except KeyError:
            pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"VRAM before load: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

# ベースモデルロード
model_name = "Qwen/Qwen3-32B"
print(f"Loading base model: {model_name}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
print("Tokenizer loaded")

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True
)
print("Base model loaded")

# LoRAアダプターロード
adapter_path = "/content/drive/MyDrive/models/qwen3-32b-poi-qlora"
print(f"\nLoading LoRA adapter from: {adapter_path}")
ft_model = PeftModel.from_pretrained(base_model, adapter_path)
ft_model.eval()
print("LoRA adapter loaded")

# Qwen3 thinking mode 無効化
original_apply = tokenizer.apply_chat_template
def patched_apply(*args, **kwargs):
    kwargs['enable_thinking'] = False
    return original_apply(*args, **kwargs)
tokenizer.apply_chat_template = patched_apply
print("\u2705 Qwen3 thinking mode disabled")

if torch.cuda.is_available():
    print(f"\nVRAM after FT model load: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

# 学習メタデータ確認
meta_path = f"{adapter_path}/training_metadata.json"
if os.path.exists(meta_path):
    with open(meta_path) as f:
        meta = json.load(f)
    print(f"\nTraining metadata:")
    print(f"  Train samples: {meta.get('train_samples')}")
    print(f"  Epochs: {meta.get('num_train_epochs')}")
    print(f"  Best eval loss: {meta.get('eval_loss_best')}")

## 3. POIデータ・ベクトルストア・テストケース読み込み

In [ ]:
from sentence_transformers import SentenceTransformer
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model_name = "intfloat/multilingual-e5-base"
print(f"Loading embedding model: {embedding_model_name}...")

embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)
print("Embedding model loaded")

In [ ]:
from geo_utils import STATIONS, enrich_all_areas

# エリア設定
areas_config = {
    "shibuya": {"name": "渋谷駅周辺", "station": STATIONS["渋谷駅"]},
    "shinjuku": {"name": "新宿駅周辺", "station": STATIONS["新宿駅"]},
    "ikebukuro": {"name": "池袋駅周辺", "station": STATIONS["池袋駅"]},
    "tokyo": {"name": "東京駅周辺", "station": STATIONS["東京駅"]},
}

# POIデータ読み込み
poi_all_file = f"{PROJECT_PATH}/data/poi_all_areas.json"
print(f"Loading POI data from {poi_all_file}...")

with open(poi_all_file, "r", encoding="utf-8") as f:
    raw_pois = json.load(f)

flat_pois = []
for poi in raw_pois:
    if "metadata" in poi:
        flat_pois.append(poi["metadata"].copy())
    else:
        flat_pois.append(poi)

print(f"Total POIs: {len(flat_pois)}")

# 空間情報付与
all_pois_enriched = enrich_all_areas(flat_pois, areas_config)
print(f"Enriched {len(all_pois_enriched)} POIs")

In [ ]:
from langchain_chroma import Chroma
from langchain_core.documents import Document

def create_documents(pois):
    docs = []
    for poi in pois:
        content = f"{poi.get('name', '')} {poi.get('category', '')} {poi.get('description', '')}"
        docs.append(Document(page_content=content, metadata=poi))
    return docs

# 全統合ベクトルストア
all_docs = create_documents(flat_pois)
vectorstore = Chroma.from_documents(
    documents=all_docs,
    embedding=embeddings,
    collection_name="pois_all"
)
print(f"Vectorstore: {len(all_docs)} documents")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
from test_cases_multi_area import ALL_MULTI_AREA_TEST_CASES, get_quick_test_cases

print(f"Total test cases: {len(ALL_MULTI_AREA_TEST_CASES)}")

# 学習データのtest_idセットを読み込み（過学習分析用）
training_data_file = f"{PROJECT_PATH}/data/phase9c_training_data.json"
with open(training_data_file, encoding="utf-8") as f:
    training_data = json.load(f)

train_test_ids = set(s["metadata"]["test_id"] for s in training_data["train"])
valid_test_ids = set(s["metadata"]["test_id"] for s in training_data["validation"])
ft_data_ids = train_test_ids | valid_test_ids

print(f"FT data test IDs: {len(ft_data_ids)} (train={len(train_test_ids)}, valid={len(valid_test_ids)})")
print(f"Non-FT test IDs: {len(ALL_MULTI_AREA_TEST_CASES) - len(ft_data_ids)}")

## 4. System A: RAG (C2) — 既存結果読み込み

In [ ]:
# C2結果読み込み
c2_results_file = f"{PROJECT_PATH}/results/phase9c_step2_20260227_071313.json"
print(f"Loading System A (C2 RAG) results from: {c2_results_file}")

with open(c2_results_file, encoding="utf-8") as f:
    c2_data = json.load(f)

system_a_raw = c2_data["systems"]["hybrid_rag"]["results"]
system_a_summary = c2_data["systems"]["hybrid_rag"]["summary"]

print(f"System A: {len(system_a_raw)} results loaded")
print(f"  Composite: {system_a_summary['overall']['avg_composite_score']}")
print(f"  Reasoning: {system_a_summary['overall']['avg_reasoning_score']}")
print(f"  Evidence: {system_a_summary['overall']['avg_evidence_score']}")
print(f"  Success%: {system_a_summary['overall']['composite_success_rate']*100:.1f}%")

## 5. System B: FT-only（RAGなし）

In [ ]:
from evaluators_multi_area import MultiAreaEvaluator

evaluator = MultiAreaEvaluator(areas_config=areas_config, all_pois=all_pois_enriched)

# System B: FT-only (システムプロンプト + 質問のみ、RAGなし)
SYSTEM_PROMPT = """あなたは東京都内の主要駅周辺エリア（渋谷駅周辺、新宿駅周辺、池袋駅周辺、東京駅周辺）の地理情報に詳しいアシスタントです。
提供されたデータに基づいて、以下の構造で回答してください。

# 回答の構造
1. **結論**: 質問への直接的な回答を最初に述べる
2. **根拠**: データから得られた具体的な証拠を引用する
3. **補足**: 注意点や不確実な点があれば述べる

# 回答ルール
- 推論過程を明示する: 「したがって」「比較すると」「分析すると」「なぜなら」等の論理接続詞を使い、結論に至る過程を示す
- 根拠を具体的に引用する: POI名、座標(緯度, 経度)、距離(m)、件数を提供データから引用し、「データから」「検索結果に基づき」等で出典を明記する
- 数値は単位付きで示す: 距離はm、件数は件、座標は(35.xxx, 139.xxx)の形式で記載する
- 比較表現を使う: 「より多い」「最も近い」「〜倍」等の比較表現で差異を明確にする
- 不確実性を正直に示す: データで確認できない点は「ただし」「データの限界として」「可能性があります」「データからは確認できません」等で明記する
- 情報がない場合は「提供データからは確認できません」と正直に回答する"""


def system_b_fn(question: str) -> dict:
    """System B: FT model + system prompt only (no RAG)"""
    from geo_utils import detect_target_area

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = ft_model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.8,
            top_k=20,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    answer = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    answer = re.sub(r'<think>.*?</think>', '', answer, flags=re.DOTALL).strip()
    answer = re.sub(r'</think>', '', answer).strip()

    del inputs, outputs
    torch.cuda.empty_cache()

    detected = detect_target_area(question, areas_config)
    return {"answer": answer, "detected_area": detected}


print("System B (FT-only) function defined")
print(f"System prompt length: {len(SYSTEM_PROMPT)} chars")

In [ ]:
# Quick test: System B
test_q = ALL_MULTI_AREA_TEST_CASES[0].prompt
print(f"Quick test Q: {test_q}")
result_b = system_b_fn(test_q)
print(f"Quick test A: {result_b['answer'][:200]}...")
print(f"Detected area: {result_b['detected_area']}")

In [ ]:
# System B: Full evaluation (130 cases)
print("="*60)
print("System B: FT-only Evaluation (130 cases)")
print("="*60)

checkpoint_b = f"{PROJECT_PATH}/results/checkpoint_c3_ft_only.json"
results_b = evaluator.evaluate_all(
    system_name="C3_ft_only",
    system_fn=system_b_fn,
    test_cases=ALL_MULTI_AREA_TEST_CASES,
    checkpoint_file=checkpoint_b
)

# スコア再計算
results_b = evaluator.recalculate_scores(results_b, ALL_MULTI_AREA_TEST_CASES)

summary_b = evaluator.generate_summary(results_b)
ob = summary_b["overall"]
print(f"\nSystem B Results:")
print(f"  Composite:  {ob['avg_composite_score']:.1f}")
print(f"  Reasoning:  {ob['avg_reasoning_score']:.2f}")
print(f"  Evidence:   {ob['avg_evidence_score']:.2f}")
print(f"  Success%:   {ob['composite_success_rate']*100:.1f}%")
print(f"  Avg Time:   {ob['avg_time_sec']:.1f}s")

gc.collect()
torch.cuda.empty_cache()
print("\n\u2705 System B evaluation complete")

## 6. System C: FT+RAG

In [ ]:
from structured_rag_system import StructuredRAGSystem
from geo_utils import detect_target_area

# FTモデルをRAGシステムに注入
print("Initializing System C (FT+RAG)...")

rag_system = StructuredRAGSystem(
    model=ft_model,
    tokenizer=tokenizer,
    vectorstore=vectorstore,
    all_pois=all_pois_enriched,
    areas_config=areas_config,
    debug=False
)
print("StructuredRAGSystem initialized with FT model")


def system_c_fn(question: str) -> dict:
    """System C: FT model + RAG"""
    result = rag_system.query(question)
    answer = result.get("answer", "")
    answer = re.sub(r'<think>.*?</think>', '', answer, flags=re.DOTALL).strip()
    answer = re.sub(r'</think>', '', answer).strip()

    detected = result.get("target_area") or detect_target_area(question, areas_config)
    return {"answer": answer, "detected_area": detected}


print("System C (FT+RAG) function defined")

In [ ]:
# Quick test: System C
test_q = ALL_MULTI_AREA_TEST_CASES[0].prompt
print(f"Quick test Q: {test_q}")
result_c = system_c_fn(test_q)
print(f"Quick test A: {result_c['answer'][:200]}...")
print(f"Detected area: {result_c['detected_area']}")

In [ ]:
# System C: Full evaluation (130 cases)
print("="*60)
print("System C: FT+RAG Evaluation (130 cases)")
print("="*60)

checkpoint_c = f"{PROJECT_PATH}/results/checkpoint_c3_ft_rag.json"
results_c = evaluator.evaluate_all(
    system_name="C3_ft_rag",
    system_fn=system_c_fn,
    test_cases=ALL_MULTI_AREA_TEST_CASES,
    checkpoint_file=checkpoint_c
)

# スコア再計算
results_c = evaluator.recalculate_scores(results_c, ALL_MULTI_AREA_TEST_CASES)

summary_c = evaluator.generate_summary(results_c)
oc = summary_c["overall"]
print(f"\nSystem C Results:")
print(f"  Composite:  {oc['avg_composite_score']:.1f}")
print(f"  Reasoning:  {oc['avg_reasoning_score']:.2f}")
print(f"  Evidence:   {oc['avg_evidence_score']:.2f}")
print(f"  Success%:   {oc['composite_success_rate']*100:.1f}%")
print(f"  Avg Time:   {oc['avg_time_sec']:.1f}s")

gc.collect()
torch.cuda.empty_cache()
print("\n\u2705 System C evaluation complete")

## 7. 3システム比較分析

In [ ]:
import numpy as np

# System A のサマリーを直接使用
oa = system_a_summary["overall"]

# === 全体比較表 ===
print("="*100)
print("3 System Comparison: Overall")
print("="*100)

metrics = [
    ("Composite Score", oa['avg_composite_score'], ob['avg_composite_score'], oc['avg_composite_score']),
    ("Reasoning Score", oa['avg_reasoning_score'], ob['avg_reasoning_score'], oc['avg_reasoning_score']),
    ("Evidence Score", oa['avg_evidence_score'], ob['avg_evidence_score'], oc['avg_evidence_score']),
    ("Success%", oa['composite_success_rate']*100, ob['composite_success_rate']*100, oc['composite_success_rate']*100),
    ("Success Rate", oa['success_rate']*100, ob['success_rate']*100, oc['success_rate']*100),
    ("Keyword Hit Rate", oa['avg_keyword_hit_rate']*100, ob['avg_keyword_hit_rate']*100, oc['avg_keyword_hit_rate']*100),
    ("Avg Time (s)", oa['avg_time_sec'], ob['avg_time_sec'], oc['avg_time_sec']),
]

print(f"{'Metric':<25} {'A: RAG(C2)':>12} {'B: FT-only':>12} {'C: FT+RAG':>12} {'B-A':>8} {'C-A':>8} {'C-B':>8}")
print("-"*100)
for name, a, b, c in metrics:
    print(f"{name:<25} {a:>12.1f} {b:>12.1f} {c:>12.1f} {b-a:>+8.1f} {c-a:>+8.1f} {c-b:>+8.1f}")

# C3目標達成判定
print(f"\n{'='*60}")
print(f"C3 Target Achievement (System C = FT+RAG)")
print(f"{'='*60}")

targets = [
    ("Composite Score", oc['avg_composite_score'], 75.0),
    ("Reasoning Score", oc['avg_reasoning_score'], 3.5),
    ("Evidence Score", oc['avg_evidence_score'], 4.0),
    ("Success%", oc['composite_success_rate']*100, 88.0),
]

all_met = True
for name, value, target in targets:
    met = value >= target
    if not met:
        all_met = False
    status = "PASS" if met else "MISS"
    print(f"  {name:<25} {value:>8.1f} / {target:.1f}  {status}")

print(f"\n  Overall: {'ALL TARGETS MET' if all_met else 'SOME TARGETS MISSED'}")

In [ ]:
# === レベル別比較 ===
print("="*100)
print("Level-wise Comparison")
print("="*100)

level_names = {1: "L1 Basic", 2: "L2 Spatial", 3: "L3 Constraint", 4: "L4 Decision", 5: "L5 Advanced"}

a_by_level = system_a_summary.get("by_level", {})
b_by_level = summary_b.get("by_level", {})
c_by_level = summary_c.get("by_level", {})

print(f"{'Level':<18} {'A: RAG(C2)':>12} {'B: FT-only':>12} {'C: FT+RAG':>12} {'B-A':>8} {'C-A':>8} {'Best':>10}")
print("-"*85)

for level in [1, 2, 3, 4, 5]:
    a_comp = a_by_level.get(level, a_by_level.get(str(level), {})).get('avg_composite_score', 0)
    b_comp = b_by_level.get(level, b_by_level.get(str(level), {})).get('avg_composite_score', 0)
    c_comp = c_by_level.get(level, c_by_level.get(str(level), {})).get('avg_composite_score', 0)
    
    best = max([(a_comp, 'A'), (b_comp, 'B'), (c_comp, 'C')], key=lambda x: x[0])
    print(f"{level_names[level]:<18} {a_comp:>12.1f} {b_comp:>12.1f} {c_comp:>12.1f} {b_comp-a_comp:>+8.1f} {c_comp-a_comp:>+8.1f} {best[1]:>10}")

In [ ]:
# === サブカテゴリ別比較 ===
print("="*100)
print("Subcategory-wise Comparison (Composite Score)")
print("="*100)

# System A のサブカテゴリ別スコアを結果から計算
from collections import defaultdict
a_by_subcat = defaultdict(list)
for r in system_a_raw:
    a_by_subcat[r['subcategory']].append(r['composite_score'])

b_by_subcat = defaultdict(list)
for r in results_b:
    b_by_subcat[r.subcategory].append(r.composite_score)

c_by_subcat = defaultdict(list)
for r in results_c:
    c_by_subcat[r.subcategory].append(r.composite_score)

all_subcats = sorted(set(list(a_by_subcat.keys()) + list(b_by_subcat.keys()) + list(c_by_subcat.keys())))

print(f"{'Subcategory':<25} {'N':>4} {'A: RAG':>10} {'B: FT':>10} {'C: FT+RAG':>10} {'C-A':>8} {'Best':>8}")
print("-"*85)

for subcat in all_subcats:
    a_scores = a_by_subcat.get(subcat, [])
    b_scores = b_by_subcat.get(subcat, [])
    c_scores = c_by_subcat.get(subcat, [])
    
    n = max(len(a_scores), len(b_scores), len(c_scores))
    a_avg = np.mean(a_scores) if a_scores else 0
    b_avg = np.mean(b_scores) if b_scores else 0
    c_avg = np.mean(c_scores) if c_scores else 0
    
    best = max([(a_avg, 'A'), (b_avg, 'B'), (c_avg, 'C')], key=lambda x: x[0])
    print(f"{subcat:<25} {n:>4} {a_avg:>10.1f} {b_avg:>10.1f} {c_avg:>10.1f} {c_avg-a_avg:>+8.1f} {best[1]:>8}")

In [ ]:
# === 寄与分離分析 ===
print("="*80)
print("Contribution Decomposition")
print("="*80)

decomp_metrics = [
    ("Composite", oa['avg_composite_score'], ob['avg_composite_score'], oc['avg_composite_score']),
    ("Reasoning", oa['avg_reasoning_score'], ob['avg_reasoning_score'], oc['avg_reasoning_score']),
    ("Evidence", oa['avg_evidence_score'], ob['avg_evidence_score'], oc['avg_evidence_score']),
    ("Success%", oa['composite_success_rate']*100, ob['composite_success_rate']*100, oc['composite_success_rate']*100),
]

print(f"{'Metric':<15} {'RAG effect':>12} {'FT effect':>12} {'Synergy':>12} {'Total(C-base)':>14}")
print(f"{'':15} {'(A-B)':>12} {'(B-A)':>12} {'(C-max)':>12} {'(C-min(A,B))':>14}")
print("-"*70)

for name, a_val, b_val, c_val in decomp_metrics:
    rag_effect = a_val - b_val    # RAG's contribution over FT-only
    ft_effect = b_val - a_val      # FT's contribution over RAG-only
    max_ab = max(a_val, b_val)
    synergy = c_val - max_ab       # Synergy beyond the best single system
    total = c_val - min(a_val, b_val)
    print(f"{name:<15} {rag_effect:>+12.2f} {ft_effect:>+12.2f} {synergy:>+12.2f} {total:>+14.2f}")

print(f"\nInterpretation:")
print(f"  RAG effect > 0: RAG provides value beyond FT alone")
print(f"  FT effect > 0: FT provides value beyond RAG alone")
print(f"  Synergy > 0: FT+RAG is better than either alone (complementary)")
print(f"  Synergy < 0: Some redundancy between FT and RAG")

In [ ]:
# === 過学習分析 ===
print("="*80)
print("Overfitting Analysis: FT-data vs Non-FT-data")
print("="*80)

for sys_label, results_list in [("B: FT-only", results_b), ("C: FT+RAG", results_c)]:
    ft_scores = [r.composite_score for r in results_list if r.test_id in ft_data_ids]
    non_ft_scores = [r.composite_score for r in results_list if r.test_id not in ft_data_ids]
    
    ft_avg = np.mean(ft_scores) if ft_scores else 0
    non_ft_avg = np.mean(non_ft_scores) if non_ft_scores else 0
    gap = ft_avg - non_ft_avg
    
    print(f"\n{sys_label}:")
    print(f"  FT data ({len(ft_scores)} cases):     avg composite = {ft_avg:.1f}")
    print(f"  Non-FT data ({len(non_ft_scores)} cases): avg composite = {non_ft_avg:.1f}")
    print(f"  Gap: {gap:+.1f}")
    
    if gap > 10:
        print(f"  \u26a0\ufe0f SIGNIFICANT overfitting detected (gap > 10pt)")
    elif gap > 5:
        print(f"  \u26a0\ufe0f Moderate overfitting (gap 5-10pt)")
    else:
        print(f"  \u2705 No significant overfitting (gap <= 5pt)")

# System Aも比較用に表示
a_ft_scores = [r['composite_score'] for r in system_a_raw if r['test_id'] in ft_data_ids]
a_non_ft_scores = [r['composite_score'] for r in system_a_raw if r['test_id'] not in ft_data_ids]
print(f"\nA: RAG(C2) (reference):")
print(f"  FT data ({len(a_ft_scores)} cases):     avg composite = {np.mean(a_ft_scores):.1f}")
print(f"  Non-FT data ({len(a_non_ft_scores)} cases): avg composite = {np.mean(a_non_ft_scores):.1f}")
print(f"  Gap: {np.mean(a_ft_scores) - np.mean(a_non_ft_scores):+.1f}")

In [ ]:
# === C2弱点改善分析 ===
print("="*80)
print("C2 Weakness Improvement: Cases with C2 composite 50-70")
print("="*80)

# C2で50-70点だったケース
weak_cases = {r['test_id']: r['composite_score'] for r in system_a_raw 
              if 50 <= r['composite_score'] < 70}

print(f"Weak cases (C2 composite 50-70): {len(weak_cases)}")

b_weak = {r.test_id: r.composite_score for r in results_b if r.test_id in weak_cases}
c_weak = {r.test_id: r.composite_score for r in results_c if r.test_id in weak_cases}

improvements_b = sum(1 for tid in weak_cases if b_weak.get(tid, 0) > weak_cases[tid])
improvements_c = sum(1 for tid in weak_cases if c_weak.get(tid, 0) > weak_cases[tid])

b_deltas = [b_weak.get(tid, 0) - weak_cases[tid] for tid in weak_cases]
c_deltas = [c_weak.get(tid, 0) - weak_cases[tid] for tid in weak_cases]

print(f"\nSystem B (FT-only):")
print(f"  Improved: {improvements_b}/{len(weak_cases)} ({improvements_b/len(weak_cases)*100:.1f}%)")
print(f"  Avg delta: {np.mean(b_deltas):+.1f}")

print(f"\nSystem C (FT+RAG):")
print(f"  Improved: {improvements_c}/{len(weak_cases)} ({improvements_c/len(weak_cases)*100:.1f}%)")
print(f"  Avg delta: {np.mean(c_deltas):+.1f}")

## 8. 可視化

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('whitegrid')

# --- Fig 1: 3 System Overall Comparison ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

systems = ['A: RAG\n(C2)', 'B: FT-only', 'C: FT+RAG']
colors = ['#3498db', '#e74c3c', '#2ecc71']

# Composite Score
comp_scores = [oa['avg_composite_score'], ob['avg_composite_score'], oc['avg_composite_score']]
bars = axes[0].bar(systems, comp_scores, color=colors, alpha=0.8)
axes[0].set_ylabel('Composite Score')
axes[0].set_title('Composite Score Comparison')
axes[0].set_ylim([0, 100])
axes[0].axhline(y=75, color='red', linestyle='--', alpha=0.5, label='C3 Target (75)')
axes[0].legend()
for bar, val in zip(bars, comp_scores):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 1, f'{val:.1f}', ha='center', fontweight='bold')

# Multi-dimensional (normalized to 0-100)
metric_names = ['Reasoning\n(x20)', 'Evidence\n(x20)', 'Success%']
a_vals = [oa['avg_reasoning_score']*20, oa['avg_evidence_score']*20, oa['composite_success_rate']*100]
b_vals = [ob['avg_reasoning_score']*20, ob['avg_evidence_score']*20, ob['composite_success_rate']*100]
c_vals = [oc['avg_reasoning_score']*20, oc['avg_evidence_score']*20, oc['composite_success_rate']*100]

x = np.arange(len(metric_names))
width = 0.25

axes[1].bar(x - width, a_vals, width, label='A: RAG(C2)', color=colors[0], alpha=0.8)
axes[1].bar(x, b_vals, width, label='B: FT-only', color=colors[1], alpha=0.8)
axes[1].bar(x + width, c_vals, width, label='C: FT+RAG', color=colors[2], alpha=0.8)

axes[1].set_ylabel('Score (0-100 scale)')
axes[1].set_title('Multi-dimensional Score Comparison')
axes[1].set_xticks(x)
axes[1].set_xticklabels(metric_names)
axes[1].legend()
axes[1].set_ylim([0, 100])

plt.tight_layout()
plt.savefig(f'{PROJECT_PATH}/results/phase9c_step3_overall_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# --- Fig 2: Level-wise Comparison ---
fig, ax = plt.subplots(figsize=(14, 6))

levels = [1, 2, 3, 4, 5]
level_labels = ['L1\nBasic', 'L2\nSpatial', 'L3\nConstraint', 'L4\nDecision', 'L5\nAdvanced']

a_scores = [a_by_level.get(l, a_by_level.get(str(l), {})).get('avg_composite_score', 0) for l in levels]
b_scores_l = [b_by_level.get(l, b_by_level.get(str(l), {})).get('avg_composite_score', 0) for l in levels]
c_scores_l = [c_by_level.get(l, c_by_level.get(str(l), {})).get('avg_composite_score', 0) for l in levels]

x = np.arange(len(levels))
width = 0.25

ax.bar(x - width, a_scores, width, label='A: RAG(C2)', color='#3498db', alpha=0.8)
ax.bar(x, b_scores_l, width, label='B: FT-only', color='#e74c3c', alpha=0.8)
ax.bar(x + width, c_scores_l, width, label='C: FT+RAG', color='#2ecc71', alpha=0.8)

ax.set_xlabel('Level')
ax.set_ylabel('Composite Score')
ax.set_title('3 System Comparison by Difficulty Level')
ax.set_xticks(x)
ax.set_xticklabels(level_labels)
ax.legend()
ax.set_ylim([0, 100])
ax.axhline(y=75, color='red', linestyle='--', alpha=0.3, label='C3 Target')

plt.tight_layout()
plt.savefig(f'{PROJECT_PATH}/results/phase9c_step3_level_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# --- Fig 3: Overfitting Analysis ---
fig, ax = plt.subplots(figsize=(10, 6))

systems_label = ['A: RAG(C2)', 'B: FT-only', 'C: FT+RAG']

# Compute FT-data vs non-FT-data scores
ft_avgs = []
non_ft_avgs = []

for results_list, is_raw in [(system_a_raw, True), (results_b, False), (results_c, False)]:
    if is_raw:
        ft = [r['composite_score'] for r in results_list if r['test_id'] in ft_data_ids]
        non_ft = [r['composite_score'] for r in results_list if r['test_id'] not in ft_data_ids]
    else:
        ft = [r.composite_score for r in results_list if r.test_id in ft_data_ids]
        non_ft = [r.composite_score for r in results_list if r.test_id not in ft_data_ids]
    ft_avgs.append(np.mean(ft) if ft else 0)
    non_ft_avgs.append(np.mean(non_ft) if non_ft else 0)

x = np.arange(len(systems_label))
width = 0.35

bars1 = ax.bar(x - width/2, ft_avgs, width, label=f'FT data ({len(ft_data_ids)} cases)', color='#e74c3c', alpha=0.8)
bars2 = ax.bar(x + width/2, non_ft_avgs, width, label=f'Non-FT data ({130-len(ft_data_ids)} cases)', color='#3498db', alpha=0.8)

ax.set_ylabel('Avg Composite Score')
ax.set_title('Overfitting Analysis: FT-data vs Non-FT-data')
ax.set_xticks(x)
ax.set_xticklabels(systems_label)
ax.legend()
ax.set_ylim([0, 100])

# Gap labels
for i in range(len(systems_label)):
    gap = ft_avgs[i] - non_ft_avgs[i]
    max_val = max(ft_avgs[i], non_ft_avgs[i])
    ax.text(i, max_val + 2, f'gap={gap:+.1f}', ha='center', fontweight='bold',
            color='red' if abs(gap) > 5 else 'black')

plt.tight_layout()
plt.savefig(f'{PROJECT_PATH}/results/phase9c_step3_overfitting.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. 結果保存

In [ ]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

def convert_to_serializable(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(i) for i in obj]
    return obj

# 結果データ構築
results_data = {
    "experiment": "phase9c_step3",
    "description": "QLoRA FineTuning evaluation (3 system comparison: RAG vs FT-only vs FT+RAG)",
    "timestamp": timestamp,
    "test_count": 130,
    "model": "Qwen/Qwen3-32B",
    "adapter": "qwen3-32b-poi-qlora",
    "training_data_count": len(ft_data_ids),
    "systems": {
        "A_rag_c2": {
            "description": "RAG (C2): Qwen3-32B + RAG (existing results)",
            "summary": convert_to_serializable(system_a_summary),
            "source": "results/phase9c_step2_20260227_071313.json",
        },
        "B_ft_only": {
            "description": "FT-only: QLoRA Qwen3-32B + system prompt (no RAG)",
            "summary": convert_to_serializable(summary_b),
            "results": [convert_to_serializable(r.to_dict()) for r in results_b],
        },
        "C_ft_rag": {
            "description": "FT+RAG: QLoRA Qwen3-32B + RAG",
            "summary": convert_to_serializable(summary_c),
            "results": [convert_to_serializable(r.to_dict()) for r in results_c],
        },
    },
    "overfitting_analysis": {
        "ft_data_ids_count": len(ft_data_ids),
        "non_ft_data_count": 130 - len(ft_data_ids),
    },
}

output_file = f"{PROJECT_PATH}/results/phase9c_step3_{timestamp}.json"
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(results_data, f, ensure_ascii=False, indent=2)

print(f"Results saved to: {output_file}")
print(f"File size: {os.path.getsize(output_file) / 1024:.1f} KB")

## 10. 結論

### 3システム比較結果

| System | Composite | Reasoning | Evidence | Success% |
|--------|-----------|-----------|----------|----------|
| A: RAG (C2) | 70.4 | 3.07 | 3.85 | 83.1% |
| B: FT-only | **TBD** | **TBD** | **TBD** | **TBD** |
| C: FT+RAG | **TBD** | **TBD** | **TBD** | **TBD** |

### C3目標達成

| 指標 | C3目標 | System C | 判定 |
|------|--------|----------|------|
| composite_score | 75+ | **TBD** | **TBD** |
| reasoning_score | 3.5+ | **TBD** | **TBD** |
| evidence_score | 4.0+ | **TBD** | **TBD** |
| composite_success_rate | 88%+ | **TBD** | **TBD** |

## 11. 結果をGitにコミット & プッシュ

In [ ]:
# 結果をプッシュ
!git -C {REPO_PATH} add notebooks/phase9c_step3_evaluation.ipynb
!git -C {REPO_PATH} add results/phase9c_step3_*
!git -C {REPO_PATH} add results/checkpoint_c3_*

!git -C {REPO_PATH} diff --cached --stat

!git -C {REPO_PATH} commit -m "results: Phase 9-C Step 3 QLoRA 3システム比較評価結果"
!git -C {REPO_PATH} push origin {BRANCH}

print("\n\u2705 評価結果をリモートブランチにプッシュしました")